# Fase 3 — Paso 3.3: Módulo Supervisado Competitivo

- Clasificación: Árbol de Decisión vs Random Forest → `estado_desaparecido` (F1-Score)
- Regresión: Árbol de Decisión vs Random Forest → `dias_solucion` (MSE, R²)

Paso 3.4: Persistencia de modelos y métricas en MongoDB.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import f1_score, mean_squared_error, r2_score

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('../../data/processed/casos_limpios.csv')

# Ajustar a las columnas reales del dataset
vars_numericas = ['edad']
vars_categoricas = ['sexo', 'provincia']
features = vars_numericas + vars_categoricas

preprocesador = ColumnTransformer([
    ('num', StandardScaler(), vars_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore'), vars_categoricas)
])

## Clasificación — `estado_desaparecido`

In [ ]:
df_clf = df.dropna(subset=features + ['estado_desaparecido'])
X = df_clf[features]
y = df_clf['estado_desaparecido']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
pipe_arbol_clf = Pipeline([
    ('prep', preprocesador),
    ('modelo', DecisionTreeClassifier(random_state=RANDOM_STATE))
])
pipe_arbol_clf.fit(X_train, y_train)
f1_arbol = f1_score(y_test, pipe_arbol_clf.predict(X_test), average='weighted')

pipe_rf_clf = Pipeline([
    ('prep', preprocesador),
    ('modelo', RandomForestClassifier(random_state=RANDOM_STATE))
])
pipe_rf_clf.fit(X_train, y_train)
f1_rf = f1_score(y_test, pipe_rf_clf.predict(X_test), average='weighted')

print(f'F1-Score Árbol de Decisión: {f1_arbol:.4f}')
print(f'F1-Score Random Forest:     {f1_rf:.4f}')

## Regresión — `dias_solucion`

In [ ]:
df_reg = df.dropna(subset=features + ['dias_solucion'])
X_r = df_reg[features]
y_r = df_reg['dias_solucion']

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_r, y_r, test_size=0.2, random_state=RANDOM_STATE
)

In [ ]:
pipe_arbol_reg = Pipeline([
    ('prep', preprocesador),
    ('modelo', DecisionTreeRegressor(random_state=RANDOM_STATE))
])
pipe_arbol_reg.fit(Xr_train, yr_train)
pred_arbol = pipe_arbol_reg.predict(Xr_test)
mse_arbol = mean_squared_error(yr_test, pred_arbol)
r2_arbol = r2_score(yr_test, pred_arbol)

pipe_rf_reg = Pipeline([
    ('prep', preprocesador),
    ('modelo', RandomForestRegressor(random_state=RANDOM_STATE))
])
pipe_rf_reg.fit(Xr_train, yr_train)
pred_rf = pipe_rf_reg.predict(Xr_test)
mse_rf = mean_squared_error(yr_test, pred_rf)
r2_rf = r2_score(yr_test, pred_rf)

print(f'Árbol  -> MSE: {mse_arbol:.2f} | R2: {r2_arbol:.4f}')
print(f'Forest -> MSE: {mse_rf:.2f} | R2: {r2_rf:.4f}')

## Paso 3.4 — Persistencia de Modelos y Trazabilidad en MongoDB

In [ ]:
# Seleccionar los mejores modelos según las métricas anteriores
mejor_modelo_clf = pipe_rf_clf if f1_rf >= f1_arbol else pipe_arbol_clf
mejor_modelo_reg = pipe_rf_reg if r2_rf >= r2_arbol else pipe_arbol_reg

joblib.dump(mejor_modelo_clf, '../../models/modelo_clasificacion.pkl')
joblib.dump(mejor_modelo_reg, '../../models/modelo_regresion.pkl')
print('Modelos guardados en /models')

In [ ]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017')
cliente = MongoClient(MONGO_URI)
db_mongo = cliente['DB_PersonasDesaparecidas_ML']
coleccion_metricas = db_mongo['metricas_modelos']

documento = {
    'fecha_entrenamiento': datetime.now().isoformat(),
    'random_state': RANDOM_STATE,
    'clasificacion': {
        'modelo_arbol': {'f1_score': f1_arbol},
        'modelo_random_forest': {'f1_score': f1_rf},
        'modelo_seleccionado': 'random_forest' if f1_rf >= f1_arbol else 'arbol_decision'
    },
    'regresion': {
        'modelo_arbol': {'mse': mse_arbol, 'r2': r2_arbol},
        'modelo_random_forest': {'mse': mse_rf, 'r2': r2_rf},
        'modelo_seleccionado': 'random_forest' if r2_rf >= r2_arbol else 'arbol_decision'
    }
}

coleccion_metricas.insert_one(documento)
print('Métricas registradas en MongoDB.')